# Final Project: Building a Data-Driven Movie Recommendation System
**Authors:** Makhabbat, Nurila, Zhaniya

## Milestone 1: Data Collection (TMDb API)

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# !!! ВАЖНО: Замените 'YOUR_TMDB_API_KEY' на ваш реальный токен TMDb API перед запуском.
# Для безопасности при отправке на GitHub рекомендуется оставлять этот плейсхолдер.
api = 'YOUR_TMDB_API_KEY'
url = 'https://api.themoviedb.org/3/discover/movie'

def fetch_movie_data(page=1):
    params = {
        'api_key': api,
        'page': page,
        'sort_by': 'popularity.desc',
        'language': 'en-US',
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return []
        
    data = response.json()
    if 'results' in data:
        return data['results']
    else:
        return []

# Сбор данных (закомментировано, так как обычно запускается один раз для сохранения в CSV)
# movies_data = []
# pages_needed = 100
# for page in range(1, pages_needed + 1):
#     movies = fetch_movie_data(page)
#     if not movies:
#         continue
#     for movie in movies:
#         if isinstance(movie, dict):
#             movie_data = {
#                 'Title': movie.get('title'),
#                 'Genre(s)': ', '.join([str(genre) for genre in movie.get('genre_ids', [])]),
#                 'Release Year': movie.get('release_date', '').split('-')[0] if movie.get('release_date') else 'N/A',
#                 'IMDB Rating': movie.get('vote_average'),
#                 'Votes': movie.get('vote_count'),
#                 'Description': movie.get('overview'),
#                 'Poster URL': f"https://image.tmdb.org/t/p/w500{movie.get('poster_path')}" if movie.get('poster_path') else None
#             }
#             movies_data.append(movie_data)
# df = pd.DataFrame(movies_data)
# df.to_csv('movies_dataset.csv', index=False)
print("Milestone 1 code defined successfully.")

## Milestone 2: Data Cleaning and Preprocessing

In [ ]:
# Для демонстрации создадим симулированный датасет, если файла еще нет на диске
try:
    df = pd.read_csv('movies_dataset.csv')
except FileNotFoundError:
    data_mock = {
        'Title': ['Movie A', 'Movie B', 'Movie C', 'Movie D', 'Movie E'],
        'Genre(s)': ['28, 12', '16, 35', '28, 53', '18', '28, 12'],
        'Release Year': [2023, 2022, np.nan, 2021, 2023],
        'IMDB Rating': [8.5, 7.2, np.nan, 6.8, 8.5],
        'Votes': [1500, 450, np.nan, 1200, 1500],
        'Description': ['Action movie adventure', 'Funny animation comedy', 'Action thriller suspense', 'Deep romantic drama', 'Action movie adventure'],
        'Poster URL': ['url1', 'url2', 'url3', 'url4', 'url1']
    }
    df = pd.DataFrame(data_mock)
    df.to_csv('movies_dataset.csv', index=False)

print("Handle missing values:")
print(df.isnull().sum())

df['IMDB Rating'] = df['IMDB Rating'].fillna(df['IMDB Rating'].mean())
df['Votes'] = df['Votes'].fillna(df['Votes'].mean())
df['Description'] = df['Description'].fillna('No Description')
df['Genre(s)'] = df['Genre(s)'].fillna('Unknown')
df['Release Year'] = df['Release Year'].fillna(df['Release Year'].median())
df['Poster URL'] = df['Poster URL'].fillna('No Poster')

numerical_columns = ['IMDB Rating', 'Votes']

# Нормализация числовых данных
scaler = StandardScaler()
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

# Кодирование категориальных признаков (жанров)
genres_encoded = df['Genre(s)'].astype(str).str.get_dummies(sep=', ').add_prefix('Genre_')
df = pd.concat([df, genres_encoded], axis=1)

# Удаление дубликатов
df.drop_duplicates(inplace=True)
df.to_csv('movies_dataset_cleaned.csv', index=False)
print("\nData cleaned and preprocessed successfully. Shape:", df.shape)

## Milestone 3: Exploratory Data Analysis (EDA)

In [ ]:
print("Statistical analysis on the dataset:")
print(df.describe())

# 1. График самых популярных жанров
genre_counts = df['Genre(s)'].astype(str).str.split(', ').explode().value_counts()
plt.figure(figsize=(12, 6))
sns.barplot(x=genre_counts.index, y=genre_counts.values, color='pink')
plt.xticks(rotation=45)
plt.title('The most common movie genres')
plt.xlabel('Genres')
plt.ylabel('Number of films')
plt.show()

# 2. Тренды выпуска фильмов
release_year_counts = df['Release Year'].value_counts().sort_index().head(70)
plt.figure(figsize=(12, 5))
sns.lineplot(x=release_year_counts.index, y=release_year_counts.values, color='green', marker='o')
plt.title('Trends in movie releases by year')
plt.xlabel('Year of issue')
plt.ylabel('Number of movies')
plt.show()

# 3. Распределение рейтингов
plt.figure(figsize=(10, 5))
sns.histplot(df['IMDB Rating'], kde=True, bins=20, color='#ff6f61')
plt.title('Distribution of IMDB ratings')
plt.xlabel('IMDB rating (Normalized)')
plt.ylabel('Frequency')
plt.show()

# 4. Тепловая карта корреляции
correlation_matrix = df[['IMDB Rating', 'Votes']].corr()
plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5)
plt.title('Correlation heatmap between numerical features')
plt.show()

## Milestone 4: Linear Regression Model for Rating Prediction

In [ ]:
df_lr = pd.read_csv('movies_dataset_cleaned.csv')

# Выделение признаков и таргета
X = df_lr.drop(columns=['IMDB Rating', 'Title', 'Description', 'Poster URL', 'Genre(s)'], errors='ignore')
y = df_lr['IMDB Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred) if len(y_test) > 1 else 0.0

print(f"Regression Metrics:")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2 score: {r2:.3f}")

# Визуализация предсказаний модели
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='#3498db')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='#e74c3c', linestyle='--', linewidth=2)
plt.title('Visualize the predicted vs. actual ratings.', fontsize=14, color='#2c3e50')
plt.xlabel('Actual IMDB rating', fontsize=11)
plt.ylabel('Predicted IMDB rating', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Milestone 5: Content-Based Recommendation System

In [ ]:
df_rec = pd.read_csv('movies_dataset_cleaned.csv')
df_rec['Genre(s)'] = df_rec['Genre(s)'].astype(str).str.replace(', ', ' ')
df_rec['Content'] = df_rec['Description'].fillna('') + ' ' + df_rec['Genre(s)']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_rec['Content'])
cosine_sim_simple = cosine_similarity(tfidf_matrix, tfidf_matrix)

def recommend_movies_simple(movie_title, cosine_sim=cosine_sim_simple):
    if movie_title not in df_rec['Title'].values:
        return f"Movie '{movie_title}' not found."
    idx = df_rec[df_rec['Title'] == movie_title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:4] # Топ-3 похожих
    movie_indices = [i[0] for i in sim_scores]
    return df_rec['Title'].iloc[movie_indices]

# Протестируем
test_movie = df_rec['Title'].iloc[0]
print(f"If you like '{test_movie}', you might also like:")
print(recommend_movies_simple(test_movie))

## Milestone 6: Database Storage (SQLite3)

In [ ]:
conn = sqlite3.connect('movies.db')
cursor = conn.cursor()
cursor.execute('DROP TABLE IF EXISTS movies')
cursor.execute('''
CREATE TABLE movies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    genre TEXT,
    release_year INTEGER,
    imdb_rating REAL,
    votes INTEGER,
    description TEXT,
    poster_url TEXT
)
''')

# Загружаем исходный датасет для сохранения "чистых" значений (до нормализации)
df_db = pd.read_csv('movies_dataset.csv')
df_db['IMDB Rating'] = df_db['IMDB Rating'].fillna(df_db['IMDB Rating'].mean())
df_db['Votes'] = df_db['Votes'].fillna(df_db['Votes'].median())
df_db['Release Year'] = df_db['Release Year'].fillna(2023)

movies_data = [
    (row['Title'], row['Genre(s)'], int(row['Release Year']), float(row['IMDB Rating']), int(row['Votes']), row['Description'], row['Poster URL'])
    for index, row in df_db.iterrows()
]

cursor.executemany('''
INSERT INTO movies (title, genre, release_year, imdb_rating, votes, description, poster_url)
VALUES (?, ?, ?, ?, ?, ?, ?)
''', movies_data)
conn.commit()

def get_all_movies():
    cursor.execute("SELECT * FROM movies LIMIT 5")
    return cursor.fetchall()

print("First 5 records inserted into SQLite database:")
for movie in get_all_movies():
    print(movie)
conn.close()

## Milestone 7: Instructions for Streamlit Launch
Код для веб-интерфейса Streamlit вынесен ниже в текстовую ячейку. Чтобы запустить веб-приложение, скопируйте этот код в отдельный файл `app.py` и выполните в терминале команду:
```bash
streamlit run app.py
```

```python
# Код для сохранения в файл app.py
import streamlit as st
import sqlite3
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_movies():
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM movies")
    movies = cursor.fetchall()
    conn.close()
    columns = ["id", "title", "genre", "release_year", "imdb_rating", "votes", "description", "poster_url"]
    return pd.DataFrame(movies, columns=columns)

def preprocess_data(df):
    df['release_year'] = df['release_year'].fillna('').astype(str)
    df['votes'] = df['votes'].fillna(0).astype(int).astype(str)
    df['imdb_rating'] = df['imdb_rating'].fillna(0).astype(float).astype(str)
    df['Content'] = (df['description'].fillna('') + ' ' + 
                     df['release_year'] + ' ' + 
                     df['votes'] + ' ' + 
                     df['imdb_rating'] + ' ' + 
                     df['genre'].fillna(''))
    return df

def recommend_movies(df, selected_movie, top_n=5):
    selected_content = df.loc[df['title'] == selected_movie, 'Content'].iloc[0]
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df['Content'].tolist() + [selected_content])
    cosine_similarities = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1]).flatten()
    df['similarity'] = cosine_similarities
    return df.sort_values(by='similarity', ascending=False).head(top_n)

st.set_page_config(page_title="Movie Recommender System", layout="wide")
st.title("Movie recommendation system from Makhabbat, Nurila, Zhaniya")
st.write("Find the perfect movie just for you based on your unique tastes!")

df_streamlit = load_movies()
preprocessed_df = preprocess_data(df_streamlit)

st.sidebar.header("User Options")
selected_movie = st.sidebar.selectbox("Choose a movie you like:", preprocessed_df['title'].tolist())

if selected_movie:
    st.subheader(f"Recommendations for: {selected_movie}")
    recommendations = recommend_movies(preprocessed_df, selected_movie)
    
    for _, row in recommendations.iterrows():
        st.markdown("---")
        col1, col2 = st.columns([1, 4])
        with col1:
            if row['poster_url'] and row['poster_url'] != 'No Poster':
                st.image(row['poster_url'], width=120)
            else:
                st.image("https://via.placeholder.com/120x180?text=No+Poster", width=120)
        with col2:
            st.markdown(f"### {row['title']}")
            st.text(f"Year: {row['release_year']}")
            st.text(f"IMDB Rating: {row['imdb_rating']}")
            st.text(f"Votes: {row['votes']}")
            st.text(f"Similarity Score: {row['similarity']:.2f}")
```